# Aetna PDF Chatbot - Example Usage

This notebook demonstrates how to use the Aetna PDF Chatbot components directly in Python code.

## Setup

First, let's import the necessary libraries and set up our environment:

In [ ]:
import os
import sys
from dotenv import load_dotenv
from openai import OpenAI
import PyPDF2
import chromadb
from chromadb.utils import embedding_functions

# Load environment variables
load_dotenv()

# Initialize OpenAI client
api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    raise ValueError("OpenAI API key not found. Please set the OPENAI_API_KEY environment variable.")
client = OpenAI(api_key=api_key)

## PDF Processing

Let's extract text from the Aetna PDF:

In [ ]:
PDF_LOCAL_PATH = "../data/pdf/ABHIL_Member_Handbook.pdf"

def extract_text_from_pdf(pdf_path=PDF_LOCAL_PATH):
    """Extract text from the PDF."""
    try:
        text = ""
        with open(pdf_path, 'rb') as file:
            pdf_reader = PyPDF2.PdfReader(file)
            for page_num in range(len(pdf_reader.pages)):
                page = pdf_reader.pages[page_num]
                page_text = page.extract_text()
                text += f"Page {page_num + 1}:\n{page_text}\n\n"
        
        return text
    except Exception as e:
        print(f"Error extracting text from PDF: {str(e)}")
        raise

# Extract text from PDF
pdf_text = extract_text_from_pdf()
print(f"Extracted {len(pdf_text)} characters from the PDF")
print(pdf_text[:500] + "...")

## Text Chunking

Now let's chunk the text into manageable pieces:

In [ ]:
import re

def chunk_text(text, chunk_size=1000, chunk_overlap=200):
    """Split the text into overlapping chunks for processing."""
    chunks = []
    
    # Split text by page markers
    pages = re.split(r'Page (\d+):', text)
    
    # Process the split result
    for i in range(1, len(pages), 2):
        if i < len(pages):
            try:
                page_num = int(pages[i])
                page_content = pages[i+1] if i+1 < len(pages) else ""
                
                if not page_content.strip():
                    continue
                
                # Further split page content into paragraphs
                paragraphs = re.split(r'\n\s*\n', page_content)
                
                current_chunk = ""
                current_chunk_metadata = {
                    "page": page_num,
                    "source": f"Page {page_num} of Aetna Better Health Illinois Member Handbook"
                }
                
                for paragraph in paragraphs:
                    paragraph = paragraph.strip()
                    if not paragraph:
                        continue
                    
                    # If adding this paragraph would exceed chunk size, save current chunk and start a new one
                    if len(current_chunk) + len(paragraph) > chunk_size and current_chunk:
                        chunks.append({
                            "text": current_chunk.strip(),
                            "metadata": current_chunk_metadata.copy()
                        })
                        
                        # Start new chunk with overlap
                        if len(current_chunk) > chunk_overlap:
                            overlap_text = current_chunk[-chunk_overlap:]
                            current_chunk = overlap_text + "\n\n" + paragraph
                        else:
                            current_chunk = paragraph
                    else:
                        # Add paragraph to current chunk
                        if current_chunk:
                            current_chunk += "\n\n" + paragraph
                        else:
                            current_chunk = paragraph
                
                # Add the last chunk from this page
                if current_chunk:
                    chunks.append({
                        "text": current_chunk.strip(),
                        "metadata": current_chunk_metadata.copy()
                    })
            except ValueError:
                continue
    
    return chunks

# Chunk the text
chunks = chunk_text(pdf_text)
print(f"Split text into {len(chunks)} chunks")
print(f"Example chunk: {chunks[0]['text'][:200]}...")
print(f"Metadata: {chunks[0]['metadata']}")

## Vector Database Creation

Now let's create a vector database from our chunks:

In [ ]:
def create_vector_db(chunks, vector_db_path="../data/vector_db"):
    """Create a vector database from text chunks."""
    # Initialize ChromaDB client
    chroma_client = chromadb.PersistentClient(path=vector_db_path)
    
    # Use OpenAI embeddings
    openai_ef = embedding_functions.OpenAIEmbeddingFunction(
        api_key=api_key,
        model_name="text-embedding-ada-002"
    )
    
    # Delete existing collection if it exists
    try:
        chroma_client.delete_collection(name="aetna_handbook")
        print("Deleted existing collection")
    except Exception:
        pass
        
    # Create new collection
    collection = chroma_client.create_collection(name="aetna_handbook", embedding_function=openai_ef)
    print("Created new vector database collection")
    
    # Add documents to collection
    ids = [f"chunk_{i}" for i in range(len(chunks))]
    texts = [chunk["text"] for chunk in chunks]
    metadatas = [chunk["metadata"] for chunk in chunks]
    
    # Add in batches to avoid timeout issues
    batch_size = 100
    for i in range(0, len(chunks), batch_size):
        end_idx = min(i + batch_size, len(chunks))
        collection.add(
            ids=ids[i:end_idx],
            documents=texts[i:end_idx],
            metadatas=metadatas[i:end_idx]
        )
        print(f"Added batch {i//batch_size + 1} to vector database")
    
    return collection

# Create vector database
collection = create_vector_db(chunks)

## Query the Vector Database

Now let's query the vector database to find relevant information:

In [ ]:
def query_vector_db(collection, query, n_results=5):
    """Query the vector database for relevant chunks."""
    results = collection.query(
        query_texts=[query],
        n_results=n_results
    )
    
    relevant_chunks = []
    for i, doc in enumerate(results['documents'][0]):
        metadata = results['metadatas'][0][i]
        relevant_chunks.append({
            "text": doc,
            "metadata": metadata
        })
    
    return relevant_chunks

# Example query
query = "What are my transportation benefits?"
relevant_chunks = query_vector_db(collection, query)

print(f"Found {len(relevant_chunks)} relevant chunks for query: '{query}'")
for i, chunk in enumerate(relevant_chunks):
    print(f"\nChunk {i+1} from {chunk['metadata']['source']}:")
    print(f"{chunk['text'][:200]}...")

## Generate Response with RAG

Finally, let's generate a response using the retrieved information:

In [ ]:
def generate_response(query, relevant_chunks):
    """Generate a response using RAG."""
    # Prepare context from relevant chunks
    context = "\n\n".join([f"Source: {chunk['metadata']['source']}\n{chunk['text']}" for chunk in relevant_chunks])
    
    # Create prompt
    prompt = f"""You are an assistant for Aetna Better Health Illinois. Answer the user's question based ONLY on the provided context. 
If the answer is not in the context, say 'I don't have information about that in the handbook.'
Be concise and accurate. Cite the page number when possible.

Context:
{context}

User Question: {query}

Answer:"""
    
    # Generate response
    response = client.chat.completions.create(
        model="gpt-4",
        messages=[
            {"role": "system", "content": "You are a helpful assistant for Aetna Better Health Illinois."},
            {"role": "user", "content": prompt}
        ],
        temperature=0.2,
        max_tokens=600
    )
    
    return response.choices[0].message.content

# Generate response
response = generate_response(query, relevant_chunks)
print(f"\nResponse to query '{query}':\n")
print(response)

## Try More Questions

Let's try a few more questions to test the system:

In [ ]:
def ask_question(question):
    """Ask a question and get a response."""
    print(f"Question: {question}")
    relevant_chunks = query_vector_db(collection, question)
    response = generate_response(question, relevant_chunks)
    print(f"\nResponse:\n{response}\n\n{'='*80}\n")
    return response

# Try some questions
questions = [
    "How do I file a grievance?",
    "What is covered under dental benefits?",
    "How can I change my primary care provider?",
    "What are the emergency services covered?"
]

for question in questions:
    ask_question(question)